# RNN — Complete Notes
**Index**
1. What is an RNN?
2. Token-by-token memory (conceptual)
3. Hidden state simulation
4. Hidden state — structured dict simulation
5. Building an RNN with Keras (Embedding + SimpleRNN + Dense)
6. Coding task — tokenise, vectorise, embed, show sentiment
7. LSTM — motivation and components

## 1. What is an RNN?

**Recurrent** means repeating. An RNN processes a sequence one token at a time and carries a **hidden state** forward at each step — that state is its memory of everything seen so far.

Classic use cases sir listed:
- Sentiment analysis
- Language translation

## 2. Token-by-token memory (conceptual)

Sir walked through "the movie was not good" token by token to show how memory accumulates:

| Token | What the RNN "remembers" |
|-------|-------------------------|
| The | sentence started |
| movie | subject is movie |
| was | waiting for description |
| not | **negation active** |
| good | good appears, but "not" changes the meaning |

Key insight: without memory of "not", the word "good" alone would suggest positive sentiment. The hidden state is what carries the negation forward.

In [ ]:
# Simple string-concatenation version of memory
# Shows the accumulation idea before introducing hidden state formally

sentence = "the movie was not good"
tokens = sentence.split()

memory = ""

for token in tokens:
    memory = memory + token + " "
    print("Current token:", token)
    print("Memory so far:", memory.strip())
    print("----------------")

## 3. Hidden State — structured simulation

The dict below mimics what a real RNN hidden state encodes: it tracks topic, negation, and sentiment word separately across the token loop. Notice how `negation_active` flips to `True` at "not" and stays True — that's exactly what the hidden state does in a trained RNN.

In [ ]:
sentence = "the movie was not good"
tokens = sentence.split()

hidden_state = {
    "topic": None,
    "negation_active": False,
    "sentiment_word": None
}

for token in tokens:
    if token in ["movie", "food", "product"]:
        hidden_state["topic"] = token

    if token == "not":
        hidden_state["negation_active"] = True

    if token in ["good", "bad", "great", "terrible"]:
        hidden_state["sentiment_word"] = token

    print("Token:", token)
    print("Hidden state:", hidden_state)
    print("-----")

# Final sentiment decision using the hidden state
if hidden_state["sentiment_word"] in ["good", "great"]:
    sentiment = "negative" if hidden_state["negation_active"] else "positive"
else:
    sentiment = "positive" if hidden_state["negation_active"] else "negative"

print("\nFinal sentiment:", sentiment)

## 4. RNN with Keras

Three layers:
- `Embedding` — converts token IDs (integers) into dense vectors of size `embedding_dim`. This is the "vector → embeddings" step.
- `SimpleRNN(32)` — 32-unit RNN layer. Processes the embedded sequence token by token, maintaining a hidden state.
- `Dense(1, activation='sigmoid')` — binary output: 0 (negative) or 1 (positive).

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

vocab_size = 1000       # 1000 most common words for the model to consider
embedding_dim = 16      # size of the word vectors
sequence_length = 8

simple_rnn_model = tf.keras.Sequential([
    layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=sequence_length),
    layers.SimpleRNN(32),           # neural network layer of 32 nodes
    layers.Dense(1, activation="sigmoid")  # 0 or 1
])

simple_rnn_model.summary()

## 5. Coding Task (assigned in class)

> Create a proper tokenisation system that takes in sentences (keep 3 sentences), vectorises these sentences, converts to embeddings, and once done shows sentiment.

Pipeline: raw text → tokenise → vectorise (word-to-index) → pad → Embedding layer → SimpleRNN → sentiment output.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# 3 sentences with labels (1 = positive, 0 = negative)
sentences = [
    "the movie was not good",
    "i absolutely loved this film",
    "it was a terrible and boring experience"
]
labels = np.array([0, 1, 0])

# Step 1: Tokenise
tokenizer = Tokenizer(num_words=1000, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)
word_index = tokenizer.word_index
print("Vocab (word -> index):", word_index)

# Step 2: Vectorise (text -> sequences of integers)
sequences = tokenizer.texts_to_sequences(sentences)
print("\nVectorised sequences:", sequences)

# Step 3: Pad to uniform length
padded = pad_sequences(sequences, maxlen=8, padding="post")
print("\nPadded sequences:\n", padded)

In [ ]:
# Step 4: Build model — Embedding converts vectors to dense embeddings, RNN processes them
model = tf.keras.Sequential([
    layers.Embedding(input_dim=1000, output_dim=16, input_length=8),
    layers.SimpleRNN(32),
    layers.Dense(1, activation="sigmoid")
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# Step 5: Train
model.fit(padded, labels, epochs=50, verbose=0)

# Step 6: Show sentiment
predictions = model.predict(padded)
for s, p in zip(sentences, predictions):
    sentiment = "Positive" if p[0] > 0.5 else "Negative"
    print(f"{sentiment} ({p[0]:.2f}) | {s}")

## 6. LSTM — Long Short-Term Memory

**Motivation (example sir gave):**

> "I grew up in France. I moved to India for college. I studied engineering. I speak fluent ___"

A vanilla RNN would likely forget "France" by the time it reaches the blank because the hidden state gets overwritten at every step. LSTM was designed to hold onto information across many more steps.

### Two states (instead of one)
- **Cell state** $C_t$ — long-term memory, runs through the sequence with minimal modification
- **Hidden state** $h_t$ — short-term working memory, same role as vanilla RNN

### Gates

**Forget gate** — decides what to delete from the cell state. Sigmoid output: 0 = forget completely, 1 = keep fully. In the France example, after "I moved to India", a trained LSTM learns to *not* forget the language information from France.

**Input gate** — decides what new information to write into the cell state.

**Output gate** — decides what part of the cell state to expose as the hidden state $h_t$ at this step.

In [ ]:
# Swapping SimpleRNN for LSTM is a one-line change
lstm_model = tf.keras.Sequential([
    layers.Embedding(input_dim=1000, output_dim=16, input_length=8),
    layers.LSTM(32),                          # replaces SimpleRNN
    layers.Dense(1, activation="sigmoid")
])

lstm_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
lstm_model.summary()

In [ ]:
# LSTM parameter count is ~4x SimpleRNN because of 4 gate weight matrices
# (forget, input, output, cell candidate)
print("SimpleRNN params:", simple_rnn_model.count_params())
print("LSTM params:      ", lstm_model.count_params())